# 🧬 Notebook Guide — AlphaFold 3 Core Architecture

## Learning Objectives
By the end of this notebook you will be able to:
- [ ] Build backbone rigid frames from N, CA, C atom coordinates (rotation + translation)
- [ ] Implement FAPE loss (Frame Aligned Point Error) from scratch
- [ ] Code Triangle Attention (row-wise and column-wise) on pair representations
- [ ] Implement Triangle Multiplicative Update (outgoing and incoming)
- [ ] Implement Outer Product Mean: MSA stack → pair representation
- [ ] Build a single Pairformer block (AF3's replacement for Evoformer)
- [ ] Compute lDDT and pLDDT numerically from predicted vs true coordinates
- [ ] Understand AF3's recycling loop and gradient-blocking strategy

## Why This Matters for AF3 Engineering
AlphaFold 3 Paper (Abramson et al. 2024) changed 3 things vs AF2:
1. **Pairformer** replaced Evoformer (same triangle operators, no MSA stack inside)
2. **Diffusion module** replaced the structure module
3. **Unified tokenization** for all biomolecule types

But the core geometric operators (FAPE, rigid frames, triangle attention) are shared with AF2.
Every AF3/computational biology ML teams Lab interview will test these.

---

## 🤖 Claude Integration — Copy These Prompts

**For Rigid Frames:**
```
Explain how AlphaFold builds backbone rigid frames from N, CA, C coordinates.
- How do you construct a rotation matrix from 3 atoms?
- What is the Gram-Schmidt procedure applied to backbone vectors?
- Why are frames 'rigid bodies'? What does it mean that they are SE(3) elements?
- Show the 4x4 homogeneous transform representation of a rigid frame.
```

**For FAPE Loss:**
```
Explain the FAPE loss in AlphaFold 2/3.
- What does 'frame-aligned' mean? Why align by predicted frames, not true frames?
- What is the d_clamp parameter and why do we clamp? (robustness to outliers)
- Show the FAPE formula: mean over frame pairs of ||x_true_in_frame - x_pred_in_frame||
- Why does FAPE enforce both local geometry (same frame) and global consistency (cross frames)?
- How does AF3's FAPE differ from AF2's? (extended to all atoms, not just backbone)
```

**For Triangle Attention:**
```
Explain triangle attention in AlphaFold's Evoformer/Pairformer.
- What is the pair representation z[i,j]? (embedding of the (i,j) residue pair)
- What does 'starting node' triangle attention compute? (attend over k using z[i,k] and z[k,j])
- What does 'ending node' triangle attention compute? (attend over k using z[k,i] and z[j,k])
- Why is this called 'triangle'? Draw the i-j-k triangle on a graph.
- How does the pair bias term gate[i,k] * z[k,j] work in the attention?
```

**For Triangle Multiplicative Update:**
```
Explain triangle multiplicative update (outgoing/incoming edges).
- Outgoing: z[i,j] += sum_k(a[i,k] * b[j,k]) — this is like outer product of row features
- Incoming: z[i,j] += sum_k(a[k,i] * b[k,j]) — this is like outer product of column features
- How does this compare to attention? (no softmax, direct multiplication)
- When would you prefer multiplicative update over attention?
```

**For lDDT/pLDDT:**
```
Explain how pLDDT is computed in AlphaFold.
- lDDT-Cα: for each residue i, average fraction of Cα-Cα distances within 15Å
  that are preserved within tolerance thresholds [0.5, 1, 2, 4]Å
- How does predicted lDDT differ from the true lDDT? (model outputs logits over 50 bins)
- What does pLDDT > 90 mean? > 70? < 50? (high, medium, low confidence regions)
- How is pLDDT used to colour AF structures in PyMol/ChimeraX?
```

---

## Architecture Cheat Sheet

```
AF3 INPUT PROCESSING:
  tokens (residues/atoms) → single repr s[i] (d_s=384)
  token pairs           → pair repr z[i,j] (d_z=128)

PAIRFORMER BLOCK (48 blocks in AF3):
  1. MSA module (optional — AF3 uses truncated MSA)
  2. Triangle attention starting node  → update z
  3. Triangle attention ending node    → update z
  4. Triangle multiplicative outgoing  → update z
  5. Triangle multiplicative incoming  → update z
  6. Pair transition (2-layer MLP)     → update z
  7. Single transition (from z → s)   → update s

DIFFUSION MODULE:
  x_noisy (all-atom coords + noise) → Conditioner → x_denoised
  Trained with denoising score matching

CONFIDENCE HEADS:
  pLDDT head: s[i] → 50-bin logits → per-residue confidence
  PAE head:   z[i,j] → 64-bin logits → NxN error matrix
  pTM head:   aggregated PAE → global TM-score estimate
```


## TL;DR — Plain English

**What this notebook does:** Walks you through the complete AlphaFold3 neural network architecture from input to output, implementing every major component with real tensor shapes you can follow.

**After this notebook you can:**
- Describe every stage of AF3: Input Embedder → Pairformer → Diffusion Module → Confidence Heads
- Implement a simplified Pairformer block with triangle attention and pair update
- Trace tensors through the network for a 50-residue protein (N=50) at each stage
- Explain what the diffusion module does differently from AF2's structure module

**Why this matters:**
- HackerRank: AlphaFold architecture questions appear in advanced bioinformatics ML assessments; being able to name components earns partial credit even without full implementations
- computational biology ML teams: This is the core interview topic — every technical screen will ask about Pairformer, triangle attention, and the diffusion head; this notebook is your primary preparation

**Time:** ~5 hours | **Difficulty:** ⭐⭐⭐⭐⭐ | **Prerequisites:** 05/01 (deep learning), 06/01 (GNNs)

## Beginner Teaching Frame

**Who should fully work through this notebook:** advanced students. Beginners should treat this notebook as a conceptual first pass.

**How to study it on a first pass:**
- aim to explain the big idea of each component
- do not get stuck on every tensor shape immediately
- learn what the block is trying to accomplish biologically and geometrically

**Common traps:** drowning in implementation detail before understanding the role of Pairformer, FAPE, PAE, or diffusion in the full system.


## Canonical Resource Upgrade

Use these in order:
- [OpenFold](https://github.com/aqlaboratory/openfold) for code orientation
- [AlphaFold 2 paper](https://www.nature.com/articles/s41586-021-03819-2)
- [AlphaFold 3 paper](https://www.nature.com/articles/s41586-024-07487-w)


## 🗺️ Prerequisites & Learning Path

| | |
|--|--|
| 🔑 Prerequisites | 05/01 (Attention mechanisms & transformers), 06/01 (Graph neural networks) |
| 📍 You Are Here | Module 07/01 — AlphaFold 3 Core Architecture |
| ➡️ Next Steps | 07/02 (AF3 data pipeline) → 07/03 (evaluation) → 07/04 (full-scale) → 10/01 (capstone) |
| 🏁 Goal | Implement and explain the Pairformer, FAPE loss, and diffusion head from AlphaFold 3 |

### 🆕 From Scratch? Start Here:
1. [Attention Is All You Need (Vaswani 2017)](https://arxiv.org/abs/1706.03762) — understand scaled dot-product attention
2. 05/01 (this repo) — attention mechanisms and transformer implementation
3. This notebook — AlphaFold 3 architecture
**Time:** 15–20h | **Difficulty:** ⭐⭐⭐⭐⭐

### 🔗 Cross-References
- Builds on: 05/01 (transformer attention), 06/01 (graph/pair representations)
- Used by: 07/02 (data pipeline feeds into this architecture), 07/03 (evaluation of outputs), 10/01 (capstone fine-tuning)

## What This Notebook Teaches (Plain English)

**AlphaFold 3** (Nature, 2024) is a computational system that predicts the 3D structure of proteins, DNA, RNA, and small molecules — and how they interact. It won its creators a share of the 2024 Nobel Prize in Chemistry.

This notebook dives into the **mathematical machinery** inside AF3. It is **advanced material** designed for people who want to implement or extend AF3, not just use it.

### Required Background

Before starting this notebook, you should understand:
- ✅ How neural networks work (layers, backpropagation, attention)
- ✅ What a protein is (amino acids, backbone atoms N-CA-C)
- ✅ Basic linear algebra (matrix multiplication, dot products)
- ✅ What AlphaFold 2 does at a high level (Evoformer, structure module)

If you're missing any of these → work through Notebooks 00 through 06 first.

### The 3 Key Innovations in AF3 vs AlphaFold 2

| Component | AlphaFold 2 | AlphaFold 3 |
|-----------|-------------|-------------|
| Structure output | Torsion angles (backbone dihedral) | Diffusion (all-atom coordinates) |
| Architecture | Evoformer (MSA + pair) | Pairformer (pair + single, no MSA stack) |
| Scope | Proteins only | Proteins + DNA + RNA + small molecules |

### Plain-English Definitions for This Notebook

| Term | Plain English |
|------|--------------|
| **Rigid frame** | Local coordinate system attached to each residue (like a car's reference frame) |
| **FAPE loss** | How far apart atoms are when measured from each residue's own frame — the training signal |
| **Triangle attention** | Attention that uses a third residue k to help compute the relationship between i and j |
| **Pairformer** | 48 stacked blocks that refine pairwise distances between all residues |
| **lDDT** | Score from 0-1 measuring how well predicted inter-atom distances match the true structure |
| **Recycling** | Running the model N times, feeding previous output back as input (like proofreading) |
| **Diffusion** | Generating 3D coordinates by gradually denoising a cloud of random points |

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# AlphaFold3 Architecture Overview — key dimensions
print("AlphaFold3 Architecture Summary")
print("=" * 50)
config = {
    "c_z (pair dim)":       128,
    "c_s (single dim)":     384,
    "c_m (MSA dim)":        64,
    "n_pairformer_blocks":  48,
    "n_heads_triangle":     4,
    "n_atoms_per_residue":  14,   # Atom14 representation
    "diffusion_steps":      200,
}
for k, v in config.items():
    print(f"  {k:<30} {v}")
print()
print("Key novelties vs AF2:")
print("  1. Pairformer replaces Evoformer (no MSA stack in later blocks)")
print("  2. Diffusion module replaces structure module (IPA)")
print("  3. Handles protein + DNA/RNA + ligand in one model")
print("  4. CCD-based ligand tokenization")
print("  5. Confidence: pTM/ipTM/PAE per pair, pLDDT per atom")

## Rigid Body Frames
### From N, CA, C coordinates → rotation matrix + translation vector

In [ ]:
import torch
import numpy as np

def make_rigid_frames(N, CA, C):
    """Compute backbone rigid frames (R, t) from N, CA, C coordinates.
    Uses Gram-Schmidt orthogonalization as in AF2 Appendix.
    """
    # Vectors within plane
    v1 = C - CA       # CA→C
    v2 = N - CA       # CA→N

    # Gram-Schmidt
    e1 = v1 / (torch.norm(v1, dim=-1, keepdim=True) + 1e-8)
    u2 = v2 - (v2 * e1).sum(-1, keepdim=True) * e1
    e2 = u2 / (torch.norm(u2, dim=-1, keepdim=True) + 1e-8)
    e3 = torch.cross(e1, e2, dim=-1)

    R = torch.stack([e1, e2, e3], dim=-1)  # (N, 3, 3) rotation matrix
    t = CA                                  # (N, 3) translation = CA position
    return R, t

def apply_rigid(R, t, x):
    """Apply rigid transformation to coordinates: x' = R @ x + t"""
    return torch.einsum('...ij,...j->...i', R, x) + t

torch.manual_seed(42)
L = 5
N  = torch.randn(L, 3)
CA = torch.randn(L, 3)
C  = torch.randn(L, 3)

R, t = make_rigid_frames(N, CA, C)
print(f"R shape: {R.shape}  (L, 3, 3)")
print(f"t shape: {t.shape}  (L, 3)")
print(f"R[0] orthogonality check (should be I): {(R[0].T @ R[0]).round(3)}")
print(f"det(R[0]) = {torch.det(R[0]):.4f}  (should be ≈ 1.0)")

x_local = torch.randn(L, 3)
x_global = apply_rigid(R, t, x_local)
print(f"\nLocal → global coords: {x_local.shape} → {x_global.shape}")

## FAPE Loss — Frame Aligned Point Error
### AF2/AF3 primary structure training loss

In [ ]:
import torch
import torch.nn.functional as F

def fape_loss(pred_frames_R, pred_frames_t, pred_coords,
              true_frames_R, true_frames_t, true_coords,
              clamp=10.0, eps=1e-8):
    """Frame Aligned Point Error (FAPE) — AF2/AF3 primary structure loss.

    For each frame i, transform all atoms into frame i's local coordinate system,
    then measure L2 distance between predicted and true positions.
    """
    L = pred_coords.shape[0]
    total = 0.0
    count = 0
    for i in range(L):
        # Transform all L atoms into frame i's local coordinates
        # local_coord = R_i^T @ (x - t_i)
        def to_local(R, t, coords):
            delta = coords - t[i:i+1]          # (L, 3)
            return torch.einsum('ij,lj->li', R[i].T, delta)  # (L, 3)

        pred_local = to_local(pred_frames_R, pred_frames_t, pred_coords)
        true_local = to_local(true_frames_R, true_frames_t, true_coords)

        diff = pred_local - true_local
        d = torch.sqrt((diff**2).sum(-1) + eps)
        d_clamped = torch.clamp(d, max=clamp)
        total += d_clamped.mean()
        count += 1

    return total / count

torch.manual_seed(42)
L = 10
# Predicted frames (slight perturbation of true)
true_R = torch.eye(3).unsqueeze(0).expand(L,-1,-1).clone()
true_t = torch.randn(L, 3) * 10
true_coords = torch.randn(L, 3) * 10

pred_t = true_t + torch.randn(L, 3) * 0.5
pred_R = true_R.clone()
pred_coords = true_coords + torch.randn(L, 3) * 0.5

loss = fape_loss(pred_R, pred_t, pred_coords, true_R, true_t, true_coords)
print(f"FAPE loss: {loss.item():.4f} Å")
print(f"FAPE is: mean over all (frame, atom) pairs of clamped L2 distance in local frame")
print(f"Clamping at 10Å prevents outlier atoms from dominating the loss")

## Triangle Attention
### The core Pairformer operator — attention over triangles of residue pairs

In [ ]:
import torch
import torch.nn as nn
import math

class TriangleAttention(nn.Module):
    """Triangle self-attention over pair representation z_ij.

    Two modes:
      'starting': for each i, attend over j-k pairs sharing node i as start
      'ending':   for each j, attend over i-k pairs sharing node j as end
    """
    def __init__(self, c_z=128, c_hidden=32, n_heads=4, mode='starting'):
        super().__init__()
        self.mode = mode
        self.n_heads = n_heads
        self.c_hidden = c_hidden
        self.norm = nn.LayerNorm(c_z)
        self.linear_q = nn.Linear(c_z, c_hidden * n_heads, bias=False)
        self.linear_k = nn.Linear(c_z, c_hidden * n_heads, bias=False)
        self.linear_v = nn.Linear(c_z, c_hidden * n_heads, bias=False)
        self.linear_b = nn.Linear(c_z, n_heads, bias=False)  # pair bias
        self.linear_g = nn.Linear(c_z, c_hidden * n_heads)   # gate
        self.linear_out = nn.Linear(c_hidden * n_heads, c_z)
        self.scale = math.sqrt(c_hidden)

    def forward(self, z, mask=None):
        # z: (B, L, L, c_z)
        B, L, L2, c = z.shape
        z_norm = self.norm(z)

        if self.mode == 'ending':
            z_norm = z_norm.transpose(1, 2)  # swap i,j for ending mode

        Q = self.linear_q(z_norm).view(B, L, L2, self.n_heads, self.c_hidden)
        K = self.linear_k(z_norm).view(B, L, L2, self.n_heads, self.c_hidden)
        V = self.linear_v(z_norm).view(B, L, L2, self.n_heads, self.c_hidden)
        b = self.linear_b(z_norm)  # (B, L, L2, n_heads) — pair bias

        # For 'starting': for each i, attend over k (the column dimension)
        # Q,K,V are (B, L_i, L_k, H, c_h)
        # attn_ijk = softmax_k( Q_ij . K_ik / sqrt(c) + b_ik )
        attn = torch.einsum('bijhd,bikhd->bijkh', Q, K) / self.scale
        attn = attn + b.unsqueeze(2)  # add pair bias b_ik
        attn = torch.softmax(attn, dim=3)  # softmax over k

        # Aggregate values
        out = torch.einsum('bijkh,bikhd->bijhd', attn, V)
        out = out.reshape(B, L, L2, self.n_heads * self.c_hidden)

        # Gate
        g = torch.sigmoid(self.linear_g(z_norm))
        out = g * out
        out = self.linear_out(out)

        if self.mode == 'ending':
            out = out.transpose(1, 2)

        return out

torch.manual_seed(42)
B, L, c_z = 1, 16, 64
z = torch.randn(B, L, L, c_z)
tri_attn = TriangleAttention(c_z=c_z, c_hidden=16, n_heads=4, mode='starting')
out = tri_attn(z)
print(f"z: {z.shape} -> out: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in tri_attn.parameters()):,}")
print("Triangle attention: O(L^3) flops, O(L^2 * c) memory")

## Triangle Multiplicative Update
### Outgoing and Incoming edge updates

In [ ]:
import torch
import torch.nn as nn

class TriangleMultiplicativeUpdate(nn.Module):
    """Triangle multiplicative update (AF2 Alg 11/12).

    Outgoing: z_ij += LayerNorm(o_ij) where o_ij = sigmoid(g_ij) * sum_k(a_ik * b_jk)
    Incoming: z_ij += LayerNorm(o_ij) where o_ij = sigmoid(g_ij) * sum_k(a_ki * b_kj)
    """
    def __init__(self, c_z=128, c_hidden=64, mode='outgoing'):
        super().__init__()
        self.mode = mode
        self.norm_in = nn.LayerNorm(c_z)
        self.linear_a = nn.Linear(c_z, c_hidden)
        self.linear_b = nn.Linear(c_z, c_hidden)
        self.linear_ga = nn.Linear(c_z, c_hidden)
        self.linear_gb = nn.Linear(c_z, c_hidden)
        self.linear_g = nn.Linear(c_z, c_z)
        self.linear_out = nn.Linear(c_hidden, c_z)
        self.norm_out = nn.LayerNorm(c_hidden)

    def forward(self, z):
        z_norm = self.norm_in(z)
        a = torch.sigmoid(self.linear_ga(z_norm)) * self.linear_a(z_norm)
        b = torch.sigmoid(self.linear_gb(z_norm)) * self.linear_b(z_norm)

        if self.mode == 'outgoing':
            # o_ij = sum_k a_ik * b_jk  →  Einstein: bik,bjk->bij
            o = torch.einsum('bikh,bjkh->bijh', a, b) / a.shape[2]**0.5
        else:  # incoming
            # o_ij = sum_k a_ki * b_kj
            o = torch.einsum('bkih,bkjh->bijh', a, b) / a.shape[1]**0.5

        o = self.norm_out(o)
        g = torch.sigmoid(self.linear_g(z_norm))
        return z + g * self.linear_out(o)

torch.manual_seed(42)
B, L, c_z = 1, 16, 64
z = torch.randn(B, L, L, c_z)
tmu_out = TriangleMultiplicativeUpdate(c_z, c_hidden=32, mode='outgoing')
tmu_in  = TriangleMultiplicativeUpdate(c_z, c_hidden=32, mode='incoming')
z = tmu_out(z)
z = tmu_in(z)
print(f"After both triangle updates: {z.shape}")
print("Outgoing: aggregates paths i→k→j (k is middle node)")
print("Incoming: aggregates paths k→i→j and k→j (alternative direction)")

## Outer Product Mean
### MSA stack → pair representation (Evoformer Algorithm 10)

In [ ]:
import torch
import torch.nn as nn

class OuterProductMean(nn.Module):
    """Outer product mean: MSA → pair representation update (AF2 Alg 10).

    For each pair (i,j): z_ij += mean_s( a_si ⊗ b_sj )
    where s indexes MSA sequences, i/j index residue positions.
    """
    def __init__(self, c_m=64, c_z=128, c_hidden=32):
        super().__init__()
        self.norm = nn.LayerNorm(c_m)
        self.linear_a = nn.Linear(c_m, c_hidden)
        self.linear_b = nn.Linear(c_m, c_hidden)
        self.linear_out = nn.Linear(c_hidden * c_hidden, c_z)

    def forward(self, m):
        # m: (B, S, L, c_m)  — MSA representation
        B, S, L, c_m = m.shape
        m = self.norm(m)
        a = self.linear_a(m)  # (B, S, L, c_h)
        b = self.linear_b(m)  # (B, S, L, c_h)
        # Outer product over sequence dimension: mean over S
        # o_bij = (1/S) * sum_s a_si ⊗ b_sj = (1/S) * a[:,:,i,:] · b[:,:,j,:]^T
        outer = torch.einsum('bsic,bsjd->bijcd', a, b) / S  # (B,L,L,c_h,c_h)
        outer = outer.reshape(B, L, L, -1)                  # flatten c_h^2
        return self.linear_out(outer)                        # (B,L,L,c_z)

torch.manual_seed(42)
B, S, L, c_m, c_z = 1, 8, 16, 32, 64  # S=8 sequences in MSA
m = torch.randn(B, S, L, c_m)
opm = OuterProductMean(c_m=c_m, c_z=c_z, c_hidden=16)
z_update = opm(m)
print(f"MSA: {m.shape} → pair update: {z_update.shape}")
print(f"Parameters: {sum(p.numel() for p in opm.parameters()):,}")
print("Outer product mean: the ONLY operation that creates pair repr from MSA")

## Pairformer Block
### AF3's replacement for Evoformer

In [ ]:
import torch
import torch.nn as nn

class PairformerBlock(nn.Module):
    """Single Pairformer block (AF3 replaces Evoformer with this).

    Operations (in order):
      1. Triangle attention (starting nodes)
      2. Triangle attention (ending nodes)
      3. Triangle multiplicative update (outgoing)
      4. Triangle multiplicative update (incoming)
      5. Pair transition (2-layer MLP per pair)
      6. Single representation update via pair (optional in AF3)
    """
    def __init__(self, c_z=128, c_s=384, n_heads=4, c_hidden=32):
        super().__init__()
        from torch import nn
        self.tri_attn_start = nn.Linear(c_z, c_z)  # simplified placeholder
        self.tri_attn_end   = nn.Linear(c_z, c_z)
        self.norm1 = nn.LayerNorm(c_z)
        self.norm2 = nn.LayerNorm(c_z)
        # Pair transition
        self.pair_trans = nn.Sequential(
            nn.LayerNorm(c_z),
            nn.Linear(c_z, c_z * 4),
            nn.ReLU(),
            nn.Linear(c_z * 4, c_z),
        )
        # Single → pair gating
        self.s_to_z = nn.Linear(c_s, c_z)
        self.single_update = nn.Sequential(
            nn.LayerNorm(c_s),
            nn.Linear(c_s, c_s),
        )

    def forward(self, z, s):
        # z: (B,L,L,c_z), s: (B,L,c_s)
        # Simplified: full version uses triangle ops defined above
        z = z + self.norm1(self.tri_attn_start(z))
        z = z + self.norm2(self.tri_attn_end(z))
        z = z + self.pair_trans(z)
        # Single repr update (uses mean of pair along one dim)
        z_mean = z.mean(dim=2)  # (B, L, c_z)
        s_gate = torch.sigmoid(self.s_to_z(s))
        z = z + s_gate.unsqueeze(2) * 0.1
        s = s + self.single_update(s)
        return z, s

torch.manual_seed(42)
B, L, c_z, c_s = 1, 16, 64, 128
z = torch.randn(B, L, L, c_z)
s = torch.randn(B, L, c_s)
block = PairformerBlock(c_z=c_z, c_s=c_s)
z_out, s_out = block(z, s)
print(f"Pairformer block:")
print(f"  z: {z.shape} → {z_out.shape}")
print(f"  s: {s.shape} → {s_out.shape}")
print(f"  Parameters: {sum(p.numel() for p in block.parameters()):,}")
print("AF3 uses 48 Pairformer blocks; each is O(L^2 * c_z) memory")

## lDDT and pLDDT — Confidence Score from Scratch

In [ ]:
import torch
import numpy as np

def compute_lddt(pred_coords, true_coords, thresholds=(0.5, 1.0, 2.0, 4.0), R=15.0):
    """Compute lDDT score (local distance difference test).

    For each residue i:
      - Find all j with true_dist(i,j) < R and |i-j| > 5
      - Count fraction of pred_dist(i,j) within threshold of true_dist(i,j)
    """
    n = pred_coords.shape[0]
    pred_dists = torch.cdist(pred_coords, pred_coords)
    true_dists = torch.cdist(true_coords, true_coords)

    per_residue = []
    for i in range(n):
        mask = (true_dists[i] < R) & (torch.arange(n) != i)
        if mask.sum() == 0:
            per_residue.append(1.0)
            continue
        diff = torch.abs(pred_dists[i][mask] - true_dists[i][mask])
        scores = [(diff < t).float().mean().item() for t in thresholds]
        per_residue.append(np.mean(scores))

    lddt = np.mean(per_residue)
    return lddt, np.array(per_residue)

torch.manual_seed(42)
L = 30
true = torch.randn(L, 3) * 10
# Good prediction: small noise
pred_good = true + torch.randn(L, 3) * 0.5
# Bad prediction: large noise
pred_bad = true + torch.randn(L, 3) * 5.0

lddt_good, per_res_good = compute_lddt(pred_good, true)
lddt_bad,  per_res_bad  = compute_lddt(pred_bad,  true)
print(f"lDDT (good prediction): {lddt_good:.3f}")
print(f"lDDT (bad prediction):  {lddt_bad:.3f}")
print(f"lDDT range: 0 (worst) → 1 (perfect)")
print(f"AF3 reports pLDDT per atom as confidence proxy")

## Recycling Loop
### Iterative refinement with gradient blocking

In [ ]:
import torch
import torch.nn as nn
import copy

class RecyclingEmbedder(nn.Module):
    """AF3 recycling: reuse previous prediction as input for next iteration."""
    def __init__(self, c_z=64, c_s=128):
        super().__init__()
        # Project recycled pair repr
        self.norm_z_recycle = nn.LayerNorm(c_z)
        self.linear_z = nn.Linear(c_z, c_z)
        # Project recycled single repr
        self.norm_s_recycle = nn.LayerNorm(c_s)
        self.linear_s = nn.Linear(c_s, c_s)
        # Bin positions (distance bins for recycled coords)
        self.n_bins = 15
        self.linear_d = nn.Linear(self.n_bins, c_z)

    def bin_distances(self, coords, min_d=2.0, max_d=22.0):
        """Encode pairwise CA distances as one-hot bin features."""
        dists = torch.cdist(coords, coords)
        bins = torch.linspace(min_d, max_d, self.n_bins + 1)
        bin_idx = torch.bucketize(dists, bins[:-1])
        return torch.nn.functional.one_hot(bin_idx.clamp(0, self.n_bins-1),
                                           self.n_bins).float()

    def forward(self, z_prev, s_prev, coords_prev):
        z_recycle = self.linear_z(self.norm_z_recycle(z_prev))
        s_recycle = self.linear_s(self.norm_s_recycle(s_prev))
        d_bins = self.bin_distances(coords_prev)
        z_recycle = z_recycle + self.linear_d(d_bins)
        return z_recycle, s_recycle

# Simulate 3 recycling iterations
torch.manual_seed(42)
B, L, c_z, c_s = 1, 20, 64, 128
recycler = RecyclingEmbedder(c_z, c_s)

z = torch.randn(B, L, L, c_z)
s = torch.randn(B, L, c_s)
coords = torch.randn(L, 3) * 10

for recycle_iter in range(3):
    z_r, s_r = recycler(z, s, coords)
    # Simulate a prediction update (placeholder for full Pairformer)
    coords = coords + torch.randn(L, 3) * 0.3
    print(f"Recycle {recycle_iter+1}: coords RMSD from init = {(coords - torch.randn(L,3)*10).norm()/L:.3f}")

print(f"\nRecycling uses torch.no_grad() for prev iterations (stop_gradient)")
print("AF3 default: 4 recycling iterations during inference")

## Interview Q&A — AF3 Architecture

In [ ]:
# AlphaFold3 Architecture — Interview Q&A
print("=" * 60)
print("ALPHAFOLD3 ARCHITECTURE — INTERVIEW Q&A")
print("=" * 60)

qas = [
    ("Q: What is the difference between Evoformer (AF2) and Pairformer (AF3)?",
     "A: AF2's Evoformer maintains a full MSA representation and updates it jointly with "
     "pair repr via outer product mean + MSA attention. AF3's Pairformer drops the MSA "
     "stack after the first few layers, operating only on pair + single representations. "
     "This reduces memory from O(S*L) to O(L^2) where S=MSA depth."),

    ("Q: Why does AF3 use diffusion instead of IPA (invariant point attention)?",
     "A: IPA predicts rigid frame transformations (backbone torsions), which struggles with "
     "ligands and nucleic acids. Diffusion models directly denoise 3D coordinates, naturally "
     "handling any atom type without per-residue frame assumptions."),

    ("Q: What is triangle attention and why O(L^3)?",
     "A: For each pair (i,j), triangle attention attends over a third residue k, so the "
     "attention pattern is a L×L×L cube. Memory-efficient implementations chunk over k "
     "to achieve O(L^2) memory at the cost of more compute passes."),

    ("Q: How does AF3 handle multi-chain complexes?",
     "A: All chains are tokenized together into one unified sequence. Protein → 1 token/residue, "
     "DNA/RNA → 1 token/nucleotide, ligand → 1 token/atom. Cross-chain attention is unrestricted "
     "in Pairformer; interface quality measured by ipTM."),

    ("Q: What is FAPE and why is it preferred over RMSD as a training loss?",
     "A: FAPE (Frame Aligned Point Error) evaluates coordinates in local frames, making it "
     "SE(3)-invariant (no need to superimpose). It is clamped at 10Å so outliers don't dominate. "
     "RMSD requires optimal superimposition (Kabsch) and is sensitive to outliers."),
]

for q, a in qas:
    print(f"\n{q}")
    print(f"{a}")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- **SE(3) rigid frames**: rotation matrix R ∈ SO(3) + translation t ∈ ℝ³ per residue — backbone coordinate system
- **Triangle inequality**: pair representation z[i,j] constrained via z[i,k] and z[k,j] (geometric consistency)
- **Scaled dot-product attention**: softmax(QKᵀ/√d_k)V — derivation from Vaswani 2017, memory O(N²)
- **FAPE loss**: Frame-Aligned Point Error — transform all atoms into each local frame, clamp at d_clamp=10Å to ignore outliers
- **DDPM / score matching**: AF3 uses diffusion over atom coordinates rather than direct regression

### 2️⃣ Must-Have Popular Resources
#### 🟢 Start Here (Before Reading AF3)
- 🆓 **AF2 explained (Yannic Kilcher)** — [youtube.com/watch?v=nGVFbPKrRWQ](https://www.youtube.com/watch?v=nGVFbPKrRWQ) — 1.5h, 1.5M views, watch BEFORE reading AF3 paper
- 🆓 **Oxford Protein Informatics AF3 walkthrough** — YouTube — academic perspective
- 🆓 **AF3 Paper Methods Section** — [nature.com/articles/s41586-024-07487-w](https://www.nature.com/articles/s41586-024-07487-w) — read Supplement 1 carefully
- 🆓 **OpenFold codebase** — [github.com/aqlaboratory/openfold](https://github.com/aqlaboratory/openfold) — study the PyTorch implementation
- 📘 AlphaFold3 Nature 2024 (Abramson et al.) — **READ the Methods section cover-to-cover**
- 📘 AlphaFold2 Nature 2021 (Jumper et al.) — essential baseline; compare Evoformer vs Pairformer
- 🎓 AF2 explained — Yannic Kilcher (YouTube, 1.5M views) — best visual intuition for the architecture
- 🎓 Oxford Protein Informatics AF3 walkthrough — technical deep-dive lecture series
- ⭐ GitHub [google-deepmind/alphafold3](https://github.com/google-deepmind/alphafold3) — official inference code
- ⭐ GitHub [aqlaboratory/openfold](https://github.com/aqlaboratory/openfold) — PyTorch trainable AF2 (8k★)

### 3️⃣ Practicals — Hands-On
- 💻 Implement triangle multiplicative update from scratch with N=32 residues, verify shapes
- 💻 Implement FAPE loss in NumPy — compute per-frame atom distances, apply clamping
- 🤗 HuggingFace Space: [facebook/esmfold_v1](https://huggingface.co/facebook/esmfold_v1) — run predictions in-browser
- 🤗 HuggingFace Space: [nferruz/ProtGPT2](https://huggingface.co/nferruz/ProtGPT2) — protein generation
- 💻 Visualize pair representations: plot z[i,j] distance heatmaps for a real PDB structure

### 4️⃣ Real-World Problems
- 🏭 computational biology ML teams — AF3 derivatives as the foundation for drug discovery pipelines
- 🏭 Insilico Medicine — target identification using predicted structures
- 🏭 Exscientia — AI-first drug design with AF3 structure prediction
- 📊 Dataset: [RCSB PDB](https://www.rcsb.org/) — 220k+ experimental structures; download mmCIF for backbone analysis
- 🔬 Application: Predicting protein–ligand binding poses for virtual screening

### 5️⃣ Interview Tips
- ❓ Must know: What changed from Evoformer → Pairformer? (removed MSA column attention; single representation dropped)
- ❓ Must know: Derive FAPE — why d_clamp=10Å? (prevents large frame-distance errors from dominating loss)
- ❓ Must know: Why does AF3 use diffusion instead of direct coordinate regression?
- ⚠️ Common mistake: Confusing pLDDT (confidence score) with actual structural accuracy
- ⚠️ Common mistake: Forgetting triangle attention is O(N²) memory — N=500 residues ≈ 250k pairs
- 💡 Pro tip: When asked about AF3 vs AF2, lead with the Pairformer/Evoformer distinction and diffusion head change

### 6️⃣ Hot Industry Topics
- 🔥 Trending: [jwohlwend/boltz](https://github.com/jwohlwend/boltz) — Boltz-2, permissive-license AF3 alternative
- 🔥 Trending: [chaidiscovery/chai-lab](https://github.com/chaidiscovery/chai-lab) — Chai-1, biomolecular structure prediction
- 🔥 Trending: NeuralPLexer3 — protein–small molecule co-folding
- 🔥 Trending: RoseTTAFold2-AA — all-atom protein + nucleic acid prediction
- 🚀 Build this: Implement a mini-Pairformer (2 layers, d_pair=32) that runs on GPU and produces pair representations for a 50-residue peptide

In [ ]:
# Resources and further reading for Module 07
print("=" * 60)
print("MODULE 07 — AlphaFold3 Core — Resources")
print("=" * 60)

resources = {
    "Primary Papers": [
        "Abramson et al. 2024 — AlphaFold3 (Nature)",
        "Jumper et al. 2021 — AlphaFold2 (Nature)",
        "Baek et al. 2021 — RoseTTAFold",
        "Lin et al. 2023 — ESMFold + ESM2",
    ],
    "Architecture Deep-Dives": [
        "AF2 Supplementary Methods — Evoformer, IPA, recycling",
        "AF3 Methods — Pairformer, diffusion, CCD",
        "Illustrated AF2 — jalammar.github.io",
    ],
    "Code Repositories": [
        "github.com/google-deepmind/alphafold3",
        "github.com/aqlaboratory/openfold",
        "github.com/facebookresearch/esm",
    ],
    "Key Evaluation Metrics": [
        "lDDT-Cα: local structural accuracy",
        "TM-score: global fold similarity",
        "DockQ: protein-protein interface quality",
        "PAE: predicted aligned error (domain orientation)",
        "pTM / ipTM: chain and interface confidence",
    ],
}
for cat, items in resources.items():
    print(f"\n{cat}:")
    for item in items:
        print(f"  * {item}")

# 🌍 Real World Problems — AF3 Architecture

## Problem 1 — Implement FAPE for a Real PDB Structure
**Dataset/Source**: PDB 7CWY (SARS-CoV-2 Spike, 1280 residues) — free download
**GitHub**: [github.com/google-deepmind/alphafold](https://github.com/google-deepmind/alphafold) — AF2 reference implementation
**Hugging Face**: [huggingface.co/spaces/simonduerr/esmfold](https://huggingface.co/spaces/simonduerr/esmfold) — ESMFold demo
**Skills**: Rigid frames, FAPE loss, backbone geometry

```python
# Download a PDB structure and compute FAPE between AF2 prediction and experimental
import urllib.request

def fetch_pdb(pdb_id):
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    with urllib.request.urlopen(url) as r:
        return r.read().decode()

def extract_backbone(pdb_text, chain='A'):
    """Extract N, CA, C coordinates from PDB text."""
    n_coords, ca_coords, c_coords = [], [], []
    for line in pdb_text.split('\n'):
        if not line.startswith('ATOM'): continue
        atom_name = line[12:16].strip()
        chain_id  = line[21:22].strip()
        if chain_id != chain: continue
        x = float(line[30:38]); y = float(line[38:46]); z = float(line[46:54])
        if atom_name == 'N':  n_coords.append([x,y,z])
        elif atom_name == 'CA': ca_coords.append([x,y,z])
        elif atom_name == 'C':  c_coords.append([x,y,z])
    return np.array(n_coords), np.array(ca_coords), np.array(c_coords)

# YOUR TASK:
# 1. pdb = fetch_pdb('6VXX')  # SARS-CoV-2 spike trimer
# 2. n, ca, c = extract_backbone(pdb, chain='A')
# 3. frames_true = [make_backbone_frame(n[i], ca[i], c[i]) for i in range(len(ca))]
# 4. Add 1Å noise to simulate a prediction: pred_ca = ca + noise
# 5. frames_pred = [make_backbone_frame(pred_n[i], pred_ca[i], pred_c[i]) for i in range(len(ca))]
# 6. fape = vectorized_fape(frames_pred, frames_true, pred_ca, ca, d_clamp=10.0)
# 7. Compare FAPE vs RMSD — which is lower? Why?
```

**Real impact**: AF3 is trained entirely on FAPE loss; RMSD is only used for evaluation. Understanding this distinction explains why AF3 can predict low-RMSD structures even when RMSD loss itself would be hard to optimize.

---

## Problem 2 — Profile Triangle Attention Memory Scaling
**Dataset/Source**: Synthetic tensors mimicking AF3 pair representations
**GitHub**: [github.com/lucidrains/alphafold2](https://github.com/lucidrains/alphafold2) — community AF2 PyTorch implementation
**Kaggle**: [Novozymes Enzyme Stability](https://www.kaggle.com/competitions/novozymes-enzyme-stability-prediction) — $50K competition
**Skills**: Triangle attention, memory profiling, AF3 engineering constraints

```python
import torch, time

def profile_triangle_attention(N_values=[32, 64, 128, 256], d_pair=64):
    """Profile memory and time scaling of triangle attention with sequence length N."""
    if not TORCH_AVAILABLE:
        print('PyTorch required for profiling')
        return
    
    results = []
    for N in N_values:
        z = torch.randn(1, N, N, d_pair)
        model = TriangleAttentionStartingNode(d_pair=d_pair, n_heads=4)
        
        # Time forward pass
        start = time.time()
        for _ in range(3):
            out = model(z)
        elapsed = (time.time() - start) / 3
        
        # Memory: O(N^2 * d_pair * n_heads) for attention matrix
        mem_mb = N * N * d_pair * 4 * 4 / 1024 / 1024  # float32
        results.append((N, elapsed*1000, mem_mb))
        print(f'  N={N:4d}: {elapsed*1000:.1f}ms, pair_repr={mem_mb:.1f}MB')
    
    # AF3 constraint: max N=2048 tokens requires chunking
    print(f'\nAF3 production: N=768 residues, d_pair=128, n_heads=16')
    print(f'Pair repr alone: {768*768*128*4/1024/1024:.0f}MB per structure')
    print('AF3 uses chunked triangle attention to stay within 40GB GPU memory')

profile_triangle_attention([32, 64, 128])
# YOUR TASK: Implement chunked triangle attention (process k in blocks of chunk_size)
# to handle N=512+ on a single 24GB GPU
```

**Real impact**: AF3's triangle attention at N=768 requires ~300MB just for the pair repr — production systems use gradient checkpointing + chunking to reduce peak memory 4-8x.

---

## Problem 3 — Build a pLDDT Calibration Curve
**Dataset/Source**: AF2 public predictions vs PDB experimental structures
**GitHub**: [github.com/google-deepmind/alphafold](https://github.com/google-deepmind/alphafold/blob/main/alphafold/common/confidence.py)
**Hugging Face**: [huggingface.co/datasets/chandar-lab/ProteinBench](https://huggingface.co/datasets/chandar-lab/ProteinBench)
**Skills**: pLDDT numerical computation, calibration, confidence metrics

```python
def plddt_calibration(pred_plddt_scores, true_lddt_scores):
    """Assess how well pLDDT predicts true lDDT — a calibration analysis.
    
    A well-calibrated model: when pLDDT=0.8, true lDDT should also be ~0.8.
    AF2/AF3 are generally well-calibrated (calibration error < 5%).
    """
    import numpy as np
    bins = np.linspace(0, 1, 11)  # 10 bins from 0 to 1
    bin_pred_mean = []
    bin_true_mean = []
    
    for i in range(len(bins)-1):
        mask = (pred_plddt_scores >= bins[i]) & (pred_plddt_scores < bins[i+1])
        if mask.sum() > 0:
            bin_pred_mean.append(pred_plddt_scores[mask].mean())
            bin_true_mean.append(true_lddt_scores[mask].mean())
    
    bin_pred_mean = np.array(bin_pred_mean)
    bin_true_mean = np.array(bin_true_mean)
    ece = np.mean(np.abs(bin_pred_mean - bin_true_mean))  # Expected Calibration Error
    return bin_pred_mean, bin_true_mean, ece

# Simulate AF2-style pLDDT scores for 1000 residues
np.random.seed(42)
pred_plddt = np.random.beta(8, 2, 1000)  # skewed towards high confidence
# True lDDT = pLDDT + small noise (well-calibrated model)
true_lddt  = pred_plddt + np.random.normal(0, 0.05, 1000)
true_lddt  = np.clip(true_lddt, 0, 1)

pred_bins, true_bins, ece = plddt_calibration(pred_plddt, true_lddt)
print(f'pLDDT calibration analysis:')
print(f'  Expected Calibration Error (ECE): {ece:.4f} (AF2 typical: 0.03-0.07)')
for p, t in zip(pred_bins, true_bins):
    bar = '█' * int(t * 20)
    print(f'  pLDDT={p:.2f}: true lDDT={t:.2f} {bar}')

# YOUR TASK: Download AF2 predictions for 100 proteins from:
# https://alphafold.ebi.ac.uk/api/search?query=human&type=organism
# Compute true lDDT against experimental PDB structures
# Plot pLDDT vs true lDDT calibration curve
```

**Real impact**: pLDDT calibration determines whether AF3's confidence scores can be trusted for drug target selection; poorly calibrated models (ECE > 0.1) lead to wasted wet lab experiments on uncertain predictions.

---

## Resources
| Resource | URL | Purpose |
|----------|-----|---------|
| AF2 Reference Code | github.com/google-deepmind/alphafold | Official AF2 implementation |
| OpenFold | github.com/aqlaboratory/openfold | Reproducible PyTorch AF2 |
| Lucidrains AF2 | github.com/lucidrains/alphafold2 | Clean community reimplementation |
| AF3 Paper | doi.org/10.1038/s41586-024-07487-w | Abramson et al. Nature 2024 |
| ESMFold HuggingFace | huggingface.co/facebook/esmfold_v1 | Single-seq structure predictor |
| ProteinBench | huggingface.co/datasets/chandar-lab/ProteinBench | Structure prediction benchmarks |
| Novozymes Kaggle | kaggle.com/competitions/novozymes-enzyme-stability-prediction | $50K stability competition |

## Mastery Check

On a first pass, success means you can:
1. explain what this notebook's component does in the larger AF3 pipeline
2. describe why geometry matters here
3. explain at least one confidence or training metric in words
4. decide whether you should revisit this notebook later for deeper implementation work


---
## 🌐 Parsing Real AlphaFold3 Output

AlphaFold3 produces a JSON confidence file and a PDB/mmCIF structure file. Learning to parse and interpret these is essential for production use.


In [ ]:
# PARSE AF3 CONFIDENCE OUTPUT
# AF3 produces a *_confidences.json file with pLDDT, PAE, pTM scores
# This cell shows the structure and how to read it

import json
import numpy as np
import matplotlib.pyplot as plt

# Simulated AF3 confidence output (matches real AF3 JSON schema exactly)
# Real file would be downloaded from https://alphafoldserver.com or internal pipeline
def generate_af3_confidence(n_residues=76, seed=42):
    """Generate realistic AF3 confidence JSON (matches actual output format)."""
    np.random.seed(seed)

    # pLDDT: per-residue, 0-100. Well-folded proteins: 80-95, loops: 40-70
    plddt = np.clip(
        np.concatenate([
            np.random.uniform(85, 97, 20),   # N-terminal helix (well-folded)
            np.random.uniform(40, 65, 8),    # loop (flexible)
            np.random.uniform(78, 92, 20),   # beta-sheet
            np.random.uniform(50, 70, 8),    # another loop
            np.random.uniform(82, 96, 20),   # C-terminal helix
        ])[:n_residues],
        0, 100
    )

    # PAE (Predicted Aligned Error): n×n matrix, lower = more confident
    # Well-defined domains: low PAE within domain, high PAE between domains
    pae = np.random.uniform(0.5, 3.0, (n_residues, n_residues))
    # Make it symmetric and zero on diagonal
    pae = (pae + pae.T) / 2
    np.fill_diagonal(pae, 0)
    # Secondary structure blocks have lower PAE (higher confidence)
    for start, end in [(0, 20), (28, 48), (56, 76)]:
        if end <= n_residues:
            pae[start:end, start:end] *= 0.3

    return {
        "plddt": plddt.tolist(),
        "pae": pae.tolist(),
        "ptm": float(np.random.uniform(0.87, 0.93)),
        "iptm": None,  # only for complexes
        "ranking_score": float(np.random.uniform(0.82, 0.91)),
        "fraction_disordered": float((plddt < 50).mean()),
    }

# Ubiquitin (76 residues) confidence
conf = generate_af3_confidence(n_residues=76)

print("AF3 CONFIDENCE FILE STRUCTURE:")
print(f"  Keys: {list(conf.keys())}")
print(f"  pLDDT: array of {len(conf['plddt'])} values (one per residue)")
print(f"  PAE: {len(conf['pae'])}×{len(conf['pae'][0])} matrix (residue pair confidence)")
print(f"  pTM: {conf['ptm']:.3f} (predicted TM-score vs true structure)")
print(f"  Fraction disordered: {conf['fraction_disordered']:.3f}")
print()

plddt = np.array(conf['plddt'])
pae = np.array(conf['pae'])

print("pLDDT INTERPRETATION:")
print("  > 90: Very high confidence (most reliably predicted)")
print("  70-90: Confident (backbone generally correct)")
print("  50-70: Low confidence (may be disordered or in error)")
print("  < 50: Very low (likely disordered — do NOT use these coordinates)")
print()
print(f"  Mean pLDDT: {plddt.mean():.1f}")
print(f"  Residues with pLDDT < 50: {(plddt < 50).sum()} ({(plddt < 50).mean()*100:.0f}%)")
print(f"  Residues with pLDDT > 90: {(plddt > 90).sum()} ({(plddt > 90).mean()*100:.0f}%)")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# pLDDT per residue
ax = axes[0]
colors = ['#d62728' if v < 50 else '#ff7f0e' if v < 70 else '#2ca02c' if v > 90 else '#1f77b4'
          for v in plddt]
ax.bar(range(len(plddt)), plddt, color=colors, alpha=0.85, width=1.0)
ax.axhline(70, color='gray', linestyle='--', alpha=0.7, label='Low conf threshold')
ax.axhline(90, color='green', linestyle='--', alpha=0.7, label='High conf threshold')
ax.set_xlabel('Residue index')
ax.set_ylabel('pLDDT')
ax.set_title('Per-Residue pLDDT\n(red<50, orange<70, blue<90, green>90)')
ax.set_ylim(0, 105)
ax.legend(fontsize=8)

# PAE heatmap
ax2 = axes[1]
im = ax2.imshow(pae, cmap='Greens_r', vmin=0, vmax=15)
plt.colorbar(im, ax=ax2, label='Predicted Aligned Error (Å)')
ax2.set_title('PAE Matrix\n(dark = confident, light = uncertain)')
ax2.set_xlabel('Residue j')
ax2.set_ylabel('Residue i')

# pLDDT distribution
ax3 = axes[2]
ax3.hist(plddt, bins=20, color='steelblue', edgecolor='white')
ax3.axvline(50, color='red', linestyle='--', label='Disordered threshold')
ax3.axvline(70, color='orange', linestyle='--', label='Low conf threshold')
ax3.axvline(90, color='green', linestyle='--', label='High conf threshold')
ax3.set_xlabel('pLDDT'); ax3.set_ylabel('Count')
ax3.set_title('pLDDT Distribution')
ax3.legend(fontsize=8)

plt.tight_layout()
plt.savefig('af3_confidence.png', dpi=100)
plt.show()

In [ ]:
# AF3 FAILURE CASE ANALYSIS
# Learn to recognize when AF3 predictions should NOT be trusted

import numpy as np
import matplotlib.pyplot as plt

# ── Case 1: Intrinsically Disordered Protein (IDP) ──
# p53 N-terminal transactivation domain (1-42): known disordered
# AF3 will predict coordinates but with very low pLDDT
itp_plddt = np.clip(np.random.normal(35, 10, 42), 10, 60)  # typical IDP pLDDT

# ── Case 2: Well-folded single domain ──
# Ubiquitin (76 aa): should have high pLDDT throughout
folded_plddt = np.clip(np.random.normal(85, 8, 76), 50, 100)

# ── Case 3: Multi-domain protein with flexible linker ──
# Two domains connected by a 20-aa linker: domains well-predicted, linker not
domain_plddt = np.concatenate([
    np.clip(np.random.normal(87, 5, 100), 60, 100),  # Domain 1
    np.clip(np.random.normal(30, 8, 20), 10, 55),    # Flexible linker
    np.clip(np.random.normal(83, 7, 80), 60, 100),   # Domain 2
])

# ── Case 4: Novel fold (no homologs in training data) ──
novel_plddt = np.clip(np.random.normal(55, 20, 90), 15, 90)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
cases = [
    (itp_plddt, "FAILURE: Intrinsically Disordered
(p53 TAD, residues 1-42)",
     "⚠️ Mean pLDDT = {:.0f} — DO NOT use coordinates!
"
     "The protein has no stable fold. AF3 will draw
"
     "arbitrary coordinates with false precision."),
    (folded_plddt, "SUCCESS: Well-Folded Single Domain
(Ubiquitin, 76 residues)",
     "✓ Mean pLDDT = {:.0f} — coordinates are reliable
"
     "High confidence throughout = well-defined fold
"
     "RMSD vs experiment typically < 1.5 Å"),
    (domain_plddt, "PARTIAL: Multi-Domain with Linker
(2 domains + 20aa linker)",
     "⚠️ Mean pLDDT = {:.0f} — domain structures OK,
"
     "but relative orientation is unreliable.
"
     "Use PAE to confirm inter-domain confidence."),
    (novel_plddt, "UNCERTAIN: Novel Fold
(no training set homologs)",
     "⚠️ Mean pLDDT = {:.0f} — moderate confidence
"
     "Check: is there any sequence homolog in PDB?
"
     "If no homolog: treat prediction with extra caution."),
]

for ax, (plddt, title, interpretation) in zip(axes.flat, cases):
    colors = ['#d62728' if v < 50 else '#ff7f0e' if v < 70 else '#2ca02c' if v > 90 else '#1f77b4'
              for v in plddt]
    ax.bar(range(len(plddt)), plddt, color=colors, alpha=0.85, width=1.0)
    ax.axhline(50, color='red', linestyle='--', alpha=0.6)
    ax.axhline(70, color='orange', linestyle='--', alpha=0.6)
    ax.set_ylim(0, 108)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Residue')
    ax.set_ylabel('pLDDT')
    ax.text(0.02, 0.03, interpretation.format(plddt.mean()),
            transform=ax.transAxes, fontsize=7.5,
            verticalalignment='bottom',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('AlphaFold3 — Four Failure/Success Patterns', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('af3_failure_analysis.png', dpi=100)
plt.show()

print("CRITICAL RULES FOR USING AF3 PREDICTIONS IN RESEARCH:")
print()
print("  1. ALWAYS check pLDDT before drawing any biological conclusion")
print("     from the predicted structure.")
print()
print("  2. Mean pLDDT < 70 for the whole protein → treat with extreme caution")
print()
print("  3. Regions with pLDDT < 50 are likely disordered. The predicted")
print("     coordinates for these regions are essentially random — they encode")
print("     only the sequence chemistry, not a real 3D fold.")
print()
print("  4. For multi-domain proteins: check PAE between domains.")
print("     Low PAE within each domain, high PAE between domains → domains")
print("     fold independently but their orientation is unreliable.")
print()
print("  5. For complexes: use ipTM (interface pTM), not overall pTM.")
print("     ipTM < 0.75 = unreliable interface prediction.")

## 🗺️ Mapping to OpenFold Source Code

The concepts implemented in this module map directly to real production code. Here is how to navigate the OpenFold codebase (if you have it cloned).


In [ ]:
# OPENFOLD CODE NAVIGATION GUIDE
# This cell shows the exact correspondence between notebook implementations
# and production OpenFold/AlphaFold code

openfold_map = {
    "OuterProductMean": {
        "notebook": "This notebook: outer_product_mean() function",
        "openfold_file": "openfold/model/evoformer.py",
        "class": "OuterProductMean",
        "key_lines": "Lines ~180-220 in the class — note how they handle chunk_size for memory",
        "paper": "AlphaFold2 (2021) Suppl. Section 1.7.3"
    },
    "TriangleAttention": {
        "notebook": "This notebook: triangle_attention() function",
        "openfold_file": "openfold/model/triangular_attention.py",
        "class": "TriangleAttentionStartingNode / TriangleAttentionEndingNode",
        "key_lines": "Two separate classes for 'starting from node' vs 'ending at node' variants",
        "paper": "AlphaFold2 (2021) Section 1.7.2, Algorithm 13/14"
    },
    "TriangleMultiplication": {
        "notebook": "This notebook: triangle_multiplicative_update() function",
        "openfold_file": "openfold/model/triangular_multiplicative_update.py",
        "class": "TriangleMultiplicationOutgoing / TriangleMultiplicationIncoming",
        "key_lines": "Note the gating mechanism (sigmoid) and layer norm placement",
        "paper": "AlphaFold2 (2021) Suppl. Algorithms 11 and 12"
    },
    "PairformerBlock": {
        "notebook": "This notebook: PairformerBlock class",
        "openfold_file": "openfold/model/evoformer.py",
        "class": "EvoformerBlock (AF2 terminology) / PairformerStack (AF3)",
        "key_lines": "One full block = TriMultOut + TriMultIn + TriAttnStart + TriAttnEnd + MLP",
        "paper": "AF3 (2024) Extended Data Fig. 2"
    },
    "FAPE": {
        "notebook": "This notebook: fape_loss() function",
        "openfold_file": "openfold/utils/loss.py",
        "class": "compute_fape()",
        "key_lines": "Note the clamping at d_clamp=10Å and the backbone_rigid_tensor usage",
        "paper": "AlphaFold2 (2021) Suppl. Section 1.9.2"
    },
    "lDDT": {
        "notebook": "This notebook: lddt() function",
        "openfold_file": "openfold/utils/loss.py",
        "class": "lddt_ca() / lddt()",
        "key_lines": "Note: OpenFold has both Cα-only and all-atom lDDT variants",
        "paper": "Mariani et al. (2013) Bioinformatics"
    },
}

print("OPENFOLD SOURCE CODE NAVIGATION GUIDE")
print("=" * 60)
print("(Assuming OpenFold cloned to ~/OpenFold or the project directory)")
print()

for concept, info in openfold_map.items():
    print(f"  {concept}")
    print(f"    Notebook: {info['notebook']}")
    print(f"    File:     {info['openfold_file']}")
    print(f"    Class:    {info['class']}")
    print(f"    Note:     {info['key_lines']}")
    print(f"    Paper:    {info['paper']}")
    print()

print("HOW TO USE THIS GUIDE:")
print("  1. Implement the concept in this notebook until you understand it")
print("  2. Open the corresponding OpenFold file")
print("  3. Find the class/function and compare your implementation")
print("  4. Note: OpenFold adds chunking, gradient checkpointing, and mixed precision")
print("     that aren't in the notebook — these are engineering optimizations, not")
print("     algorithmic changes")
print()
print("OPENFOLD INSTALLATION:")
print("  git clone https://github.com/aqlaboratory/openfold")
print("  pip install -e openfold/")
print("  # Then navigate the src/ directory with this map")

## ✏️ Exercise: Implement the AF3 Input Embedder From Scratch

AlphaFold3's input embedder converts a raw protein sequence into a per-residue feature tensor of shape `(N, c_s)`. This is the very first module in the AF3 forward pass.

**Your task:**
1. Build `InputEmbedder` using `nn.Embedding` (for token index) and `nn.Linear` (to project to `c_s` dimensions).
2. Forward pass: embed each token, project, add a learned position bias.
3. Verify that the output shape is `(N, c_s)` for `N=50`, `c_s=384`.

**Expected output:**
```
input_embedder output shape: torch.Size([50, 384])  ✓
```

**Hint:** AF3 uses 32 token types (20 amino acids + gap + 11 special tokens). Use `num_embeddings=32` for `nn.Embedding`.

In [ ]:
import torch
import torch.nn as nn

# ── Constants (from AF3 paper, Table 1) ──────────────────────────────
NUM_TOKENS = 32   # 20 AA + gap + 11 special
C_S        = 384  # single representation dimension
N_RES      = 50   # sequence length for this exercise


class InputEmbedder(nn.Module):
    """Maps token indices to (N, c_s) single representations."""

    def __init__(self, num_tokens: int = NUM_TOKENS, c_s: int = C_S):
        super().__init__()
        # TODO 1: create self.token_embed = nn.Embedding(num_tokens, c_s)
        # TODO 2: create self.pos_embed   = nn.Embedding(512, c_s)
        #         (512 is a safe max sequence length)
        # TODO 3: create self.layer_norm  = nn.LayerNorm(c_s)
        pass

    def forward(self, token_idx: torch.Tensor) -> torch.Tensor:
        """
        Args:
            token_idx: (N,) integer tensor of amino acid indices
        Returns:
            s: (N, c_s) float tensor
        """
        # TODO 4: compute position indices:  pos = torch.arange(len(token_idx))
        # TODO 5: s = self.token_embed(token_idx) + self.pos_embed(pos)
        # TODO 6: return self.layer_norm(s)
        pass


# ── Test ──────────────────────────────────────────────────────────────
torch.manual_seed(0)
model = InputEmbedder()
token_idx = torch.randint(0, NUM_TOKENS, (N_RES,))
output = model(token_idx)

if output is None:
    print('Implement the forward pass first.')
else:
    expected = torch.Size([N_RES, C_S])
    status = '✓' if output.shape == expected else '✗'
    print(f'input_embedder output shape: {output.shape}  {status}')
    if output.shape != expected:
        print(f'Expected: {expected}')
    else:
        print(f'Mean activation: {output.mean().item():.4f} '
              f'(should be near 0 after LayerNorm)')

# ── SOLUTION (uncomment to check) ──────────────────────────────────────
# class InputEmbedder(nn.Module):
#     def __init__(self, num_tokens=NUM_TOKENS, c_s=C_S):
#         super().__init__()
#         self.token_embed = nn.Embedding(num_tokens, c_s)
#         self.pos_embed   = nn.Embedding(512, c_s)
#         self.layer_norm  = nn.LayerNorm(c_s)
#     def forward(self, token_idx):
#         pos = torch.arange(len(token_idx), device=token_idx.device)
#         s = self.token_embed(token_idx) + self.pos_embed(pos)
#         return self.layer_norm(s)


---
## 🐛 Debug Exercise — Pairformer Architecture Bugs

Find and fix **3 bugs** in this simplified Pairformer block.

<details><summary>Show bugs</summary>

**Bug 1:** Triangle update multiplies `z[i,j]` by all `z[i,k] * z[k,j]`, but the outer product sum should be a dot product over the hidden dimension, not element-wise multiply then sum over k. Use `einsum('ikh,kjh->ijh', a, b)`.

**Bug 2:** Pair bias in attention is added to Q before softmax, but should be added to the attention LOGITS (after Q@K^T, before softmax).

**Bug 3:** The layer norm in the Pairformer is applied after the residual connection: `z = layernorm(z + update)`. Correct pre-norm: `z = z + layernorm_then_update(z)`.

</details>

In [ ]:
import torch
import torch.nn as nn

class BuggyPairformerBlock(nn.Module):
    def __init__(self, c_z=32, n_heads=4):
        super().__init__()
        self.c_z = c_z
        self.n_heads = n_heads
        self.head_dim = c_z // n_heads
        self.to_q = nn.Linear(c_z, c_z)
        self.to_k = nn.Linear(c_z, c_z)
        self.to_v = nn.Linear(c_z, c_z)
        self.pair_bias = nn.Linear(c_z, n_heads, bias=False)
        self.out = nn.Linear(c_z, c_z)
        self.norm = nn.LayerNorm(c_z)

    def triangle_update(self, z):
        # z: (N, N, c_z)
        a = self.to_q(z)  # (N,N,c_z)
        b = self.to_k(z)
        # BUG 1: element-wise multiply then sum over N, not einsum dot product
        update = (a.unsqueeze(1) * b.unsqueeze(0)).sum(-1)  # shape wrong
        return update

    def attention(self, z):
        N = z.shape[0]
        Q = self.to_q(z).reshape(N, N, self.n_heads, self.head_dim).permute(2,0,1,3)
        K = self.to_k(z).reshape(N, N, self.n_heads, self.head_dim).permute(2,0,1,3)
        V = self.to_v(z).reshape(N, N, self.n_heads, self.head_dim).permute(2,0,1,3)
        bias = self.pair_bias(z).permute(2,0,1).unsqueeze(-1)  # (heads,N,N,1)
        # BUG 2: bias added to Q before attention logits, should add to Q@K.T
        Q = Q + bias  # wrong: shape mismatch and wrong position
        logits = (Q @ K.transpose(-2,-1)) / (self.head_dim**0.5)
        attn = torch.softmax(logits, dim=-1)
        out = (attn @ V).permute(1,2,0,3).reshape(N, N, self.c_z)
        return self.out(out)

    def forward(self, z):
        # BUG 3: post-norm (norm after residual) — should be pre-norm
        z = self.norm(z + self.attention(z))  # should be z + f(norm(z))
        return z

N, c_z = 8, 32
z = torch.randn(N, N, c_z)
block = BuggyPairformerBlock(c_z=c_z)
try:
    out = block(z)
    print(f"Output shape: {out.shape}  (expected: ({N},{N},{c_z}))")
    print("Shape OK but computation is wrong (bugs 1 and 3 are logical, not shape errors)")
except Exception as e:
    print(f"Error (from bug 2 shape mismatch): {e}")
print("Bug 1: triangle_update output shape is wrong due to element-wise * then sum")
print("Bug 2: Q + bias fails (N,N,h,d) + (h,N,N,1) — should add bias to logits")
print("Bug 3: post-norm reduces gradient flow compared to pre-norm (AF3 uses pre-norm)")
